## Read MLS raw data
Raw MLS data contains formatting inconsistencies, missing values, and fields that need transformation
before analysis. This phase prepares the dataset for reliable analytics.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import csv

In [3]:
# Load the dataset
sold = pd.read_csv('sold.csv')

/var/folders/fb/y3cbnxzj3rs43kpb74nk2f0r0000gn/T/ipykernel_18113/3282145019.py:2: DtypeWarning: Columns (0,1,4,9,78,79,80,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('sold.csv')


In [4]:
sold_cleaned = sold.copy()
sold_cleaned.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,94401,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,92394,NaN,52320.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,93240,NaN,217364.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,92308,NaN,217800.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,93544,0.0,108883.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [5]:
sold_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591733 entries, 0 to 591732
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 520175 non-null  object 
 1   ListAgentAOR                  523685 non-null  object 
 2   Flooring                      347801 non-null  object 
 3   ViewYN                        532453 non-null  object 
 4   WaterfrontYN                  323 non-null     object 
 5   BasementYN                    9760 non-null    object 
 6   PoolPrivateYN                 517524 non-null  object 
 7   OriginalListPrice             590013 non-null  float64
 8   ListingKey                    591733 non-null  int64  
 9   ListAgentEmail                547714 non-null  object 
 10  CloseDate                     591733 non-null  object 
 11  ClosePrice                    591726 non-null  float64
 12  ListAgentFirstName            588300 non-nul

### Convert date fields to datetime format

In [6]:
date_columns = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
for col in date_columns:
    sold_cleaned[col] = pd.to_datetime(sold_cleaned[col], errors='coerce')

In [7]:
# sold_cleaned.info()

### Remove unnecessary or redundant columns
Here we only retain core analysis columns with low missing ratio 

In [8]:
sold_retain = [
    'ClosePrice', 'CloseDate', 'OriginalListPrice', 'ListPrice',
    'DaysOnMarket', 'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
    'BathroomsTotalInteger', 'YearBuilt', 'City', 'PostalCode',
    'CountyOrParish', 'StateOrProvince', 'Latitude', 'Longitude',
    'PropertyType', 'PropertySubType',
    'ListingContractDate', 'PurchaseContractDate', 'ContractStatusChangeDate'
]

sold_cleaned = sold_cleaned[sold_retain]
sold_cleaned.head()

,ClosePrice,CloseDate,OriginalListPrice,ListPrice,DaysOnMarket,LivingArea,LotSizeAcres,BedroomsTotal,BathroomsTotalInteger,YearBuilt,...,PostalCode,CountyOrParish,StateOrProvince,Latitude,Longitude,PropertyType,PropertySubType,ListingContractDate,PurchaseContractDate,ContractStatusChangeDate
0,240000.0,2024-01-26,499000.0,295000.0,777,1140.0,NaN,2.0,2.0,1988.0,...,94401,San Mateo,CA,NaN,NaN,Residential,Condominium,2021-10-06,2023-11-22,2024-01-26
1,950.0,2024-01-24,0.0,808.0,901,NaN,1.2000,NaN,NaN,1980.0,...,92394,San Bernardino,CA,34.529407,-117.319674,CommercialLease,Retail,2021-08-06,2024-01-24,2024-01-24
2,45000.0,2024-01-16,75000.0,75000.0,865,NaN,4.9900,NaN,NaN,NaN,...,93240,Kern,CA,35.618009,-118.473141,Land,NaN,2021-07-18,2023-12-22,2024-01-16
3,141500.0,2024-01-08,199000.0,145000.0,830,NaN,5.0000,NaN,NaN,NaN,...,92308,San Bernardino,CA,34.396733,-117.220954,Land,NaN,2021-07-14,2023-10-24,2024-01-08
4,15000.0,2024-01-17,19500.0,19500.0,805,NaN,2.4996,NaN,NaN,NaN,...,93544,Los Angeles,CA,34.454756,-117.782248,Land,NaN,2021-06-07,2023-08-25,2024-01-17


### Missing Value Treatment

In [9]:
# highmissing threshold
p_high = 0.5
p_very_high = 0.9

sold_missing_summary = pd.DataFrame({
    'missing_count': sold_cleaned.isna().sum(),
    'missing_ratio': sold_cleaned.isna().mean()
}).sort_values('missing_ratio', ascending=False)

sold_high_missing_summary = sold_missing_summary[sold_missing_summary['missing_ratio'] > p_high].copy()

sold_high_missing_summary['extreme_high_missing_flag'] = (sold_high_missing_summary['missing_ratio'] > p_very_high).astype(int)

sold_high_missing_summary

,missing_count,missing_ratio,extreme_high_missing_flag


Based on the result, it since that we have already filtered columns with high missing ratios

Fill NaN for missing value

In [10]:
sold_cleaned.fillna(np.nan, inplace=True)


### Ensure numeric fields are properly typed

In [11]:
sold_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591733 entries, 0 to 591732
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ClosePrice                591726 non-null  float64       
 1   CloseDate                 591733 non-null  datetime64[ns]
 2   OriginalListPrice         590013 non-null  float64       
 3   ListPrice                 590823 non-null  float64       
 4   DaysOnMarket              591733 non-null  int64         
 5   LivingArea                549951 non-null  float64       
 6   LotSizeAcres              535701 non-null  float64       
 7   BedroomsTotal             552324 non-null  float64       
 8   BathroomsTotalInteger     564309 non-null  float64       
 9   YearBuilt                 566783 non-null  float64       
 10  City                      591293 non-null  object        
 11  PostalCode                591568 non-null  object        
 12  Co

In [12]:
pd.set_option('display.max_columns', None)
sold_cleaned.head()

,ClosePrice,CloseDate,OriginalListPrice,ListPrice,DaysOnMarket,LivingArea,LotSizeAcres,BedroomsTotal,BathroomsTotalInteger,YearBuilt,City,PostalCode,CountyOrParish,StateOrProvince,Latitude,Longitude,PropertyType,PropertySubType,ListingContractDate,PurchaseContractDate,ContractStatusChangeDate
0,240000.0,2024-01-26,499000.0,295000.0,777,1140.0,NaN,2.0,2.0,1988.0,San Mateo,94401,San Mateo,CA,NaN,NaN,Residential,Condominium,2021-10-06,2023-11-22,2024-01-26
1,950.0,2024-01-24,0.0,808.0,901,NaN,1.2000,NaN,NaN,1980.0,Victorville,92394,San Bernardino,CA,34.529407,-117.319674,CommercialLease,Retail,2021-08-06,2024-01-24,2024-01-24
2,45000.0,2024-01-16,75000.0,75000.0,865,NaN,4.9900,NaN,NaN,NaN,Lake Isabella,93240,Kern,CA,35.618009,-118.473141,Land,NaN,2021-07-18,2023-12-22,2024-01-16
3,141500.0,2024-01-08,199000.0,145000.0,830,NaN,5.0000,NaN,NaN,NaN,Apple Valley,92308,San Bernardino,CA,34.396733,-117.220954,Land,NaN,2021-07-14,2023-10-24,2024-01-08
4,15000.0,2024-01-17,19500.0,19500.0,805,NaN,2.4996,NaN,NaN,NaN,Llano,93544,Los Angeles,CA,34.454756,-117.782248,Land,NaN,2021-06-07,2023-08-25,2024-01-17


#### Change BatheroomsTotalInteger and YearBuilt Column to int and keep missing values at the same time

In [13]:
sold_cleaned['BathroomsTotalInteger'] = sold_cleaned['BathroomsTotalInteger'].astype('Int64')
sold_cleaned['YearBuilt'] = sold_cleaned['YearBuilt'].astype('Int64')

### Remove or flag invalid numeric values

In [14]:
# filter with numeric columns and check for negative values
numeric_cols = sold_cleaned.select_dtypes(include=['int64', 'float64', 'Int64']).columns

negative_summary = []

for col in numeric_cols:
    negative_count = (sold_cleaned[col] < 0).sum()
    
    # Only include columns with negative values in the summary
    if negative_count > 0:
        negative_summary.append({
            'column': col,
            'negative_count': negative_count,
            'min_value': sold_cleaned[col].min()
        })

negative_summary = pd.DataFrame(negative_summary)

negative_summary

,column,negative_count,min_value
0,DaysOnMarket,60,-288.000000
1,Latitude,5,-117.472493
2,Longitude,572770,-177.646696


Latitude and Longitude can be negative so invalid values only happen in DaysOnMarket

In [15]:
# drop rows with negative values in 'DaysOnMarket'
sold_cleaned = sold_cleaned[sold_cleaned['DaysOnMarket'] >= 0].copy()
(sold_cleaned['DaysOnMarket'] < 0).sum()

np.int64(0)

## Data Consistency Check 
Validate the logical order of date fields: ListingContractDate should precede PurchaseContractDate,
which should precede CloseDate. Create boolean flag columns to mark records that violate these rules:
-  listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag

In [16]:
sold_cleaned['listing_after_purchase_flag'] = (sold_cleaned['ListingContractDate'] > sold_cleaned['PurchaseContractDate'])
sold_cleaned['listing_after_purchase_flag'].value_counts()
sold_cleaned['purchase_after_close_flag'] = (sold_cleaned['PurchaseContractDate'] > sold_cleaned['CloseDate'])
sold_cleaned['purchase_after_close_flag'].value_counts()
sold_cleaned['negative_timeline_flag'] = sold_cleaned['listing_after_purchase_flag'] | sold_cleaned['purchase_after_close_flag']
sold_cleaned['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    590938
True        735
Name: count, dtype: int64

## Geographic Data Checks
- Flag records with missing coordinates (Latitude or Longitude is null)
- Flag Latitude = 0 or Longitude = 0 (sentinel null values)
- Flag Longitude > 0 errors (California coordinates should be negative)
- Flag out-of-state or implausible coordinates

In [17]:
# either latitude or longitude is missing
sold_cleaned['missing_coordinates_flag'] = sold_cleaned['Latitude'].isna() | sold_cleaned['Longitude'].isna()
sold_cleaned['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    572643
True      19030
Name: count, dtype: int64

In [18]:
# either latitude or longitude == 0
sold_cleaned['zero_coordinates_flag'] = (sold_cleaned['Latitude'] == 0) | (sold_cleaned['Longitude'] == 0)
sold_cleaned['zero_coordinates_flag'].value_counts()

zero_coordinates_flag
False    591630
True         43
Name: count, dtype: int64

In [19]:
# longitude > 0 (should be negative in the CA)
sold_cleaned['invalid_longitude_flag'] = sold_cleaned['Longitude'] > 0
sold_cleaned['invalid_longitude_flag'].value_counts()

invalid_longitude_flag
False    591611
True         62
Name: count, dtype: int64

Realize that some transactions are from other State and should be deal with

In [20]:
sold_cleaned['StateOrProvince'].value_counts()

StateOrProvince
CA    591596
AZ        16
NV         8
OK         7
TX         6
OS         6
FL         5
HI         4
MO         3
CO         3
ID         3
NY         2
AR         2
VA         1
NJ         1
OR         1
MT         1
ME         1
TN         1
MI         1
GA         1
AL         1
OH         1
LA         1
NM         1
Name: count, dtype: int64

In [21]:
sold_cleaned['implausible_coordinate_flag'] = (
    (sold_cleaned['Latitude'] < 32) |
    (sold_cleaned['Latitude'] > 42) |
    (sold_cleaned['Longitude'] < -125) |
    (sold_cleaned['Longitude'] > -114)
)
sold_cleaned['implausible_coordinate_flag'].value_counts()

implausible_coordinate_flag
False    591487
True        186
Name: count, dtype: int64

## Output a cleaned data

In [22]:
sold_cleaned.to_csv('sold_cleaned.csv', index=False) 

In [23]:
sold_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 591673 entries, 0 to 591732
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ClosePrice                   591666 non-null  float64       
 1   CloseDate                    591673 non-null  datetime64[ns]
 2   OriginalListPrice            589953 non-null  float64       
 3   ListPrice                    590763 non-null  float64       
 4   DaysOnMarket                 591673 non-null  int64         
 5   LivingArea                   549896 non-null  float64       
 6   LotSizeAcres                 535643 non-null  float64       
 7   BedroomsTotal                552269 non-null  float64       
 8   BathroomsTotalInteger        564254 non-null  Int64         
 9   YearBuilt                    566725 non-null  Int64         
 10  City                         591233 non-null  object        
 11  PostalCode                   59